# A8: RAG system for DataPact

INFO 490 MG, Team 15.

RAG means we look up the right text first, then give it to the model with the question. The steps are:

Data → Embeddings → Retrieval → LLM → Output

We test 3 embedding models with 3 ways of chunking, so 9 setups in total. The generation model
never changes. We use SmolLM2-1.7B-Instruct every time because it was our pick in A6 and our
Django app already runs it.

**How to run:** the steps are in `README.md`. Set `RUN_MODE` in the next cell first.

| RUN_MODE | What it does |
| --- | --- |
| `full` | Real models and all the queries. This is the real run. |
| `quick` | Real models and 2 queries. Run this first to make sure the GPU works. |
| `dry` | Fake models with no downloads. This only checks that the code runs, so ignore its numbers. |

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

RUN_MODE = os.environ.get("RAG_RUN_MODE", "full")  # full, quick or dry

BASE = Path("rag_system") if Path("rag_system").is_dir() else Path(".")

# Ashok's documents go in knowledge_base/. Until then the small sample set is used.
USING_SAMPLE = not any((BASE / "knowledge_base").glob("*.md"))
KB_DIR = BASE / ("sample_kb" if USING_SAMPLE else "knowledge_base")
QUERY_FILE = BASE / ("sample_queries.json" if USING_SAMPLE else "test_queries.json")

# Only a full run on the real documents writes to results/. Anything else stays out of the way.
if RUN_MODE == "full" and not USING_SAMPLE:
    OUT_DIR = BASE / "results"
else:
    OUT_DIR = BASE / f"results_{RUN_MODE}{'_sample' if USING_SAMPLE else ''}"
OUT_DIR.mkdir(exist_ok=True)

print("Run mode:", RUN_MODE)
print("Knowledge base:", KB_DIR)
print("Saving to:", OUT_DIR)
if USING_SAMPLE:
    print("NOTE: using the sample documents. These results are not for the report.")

## Part 1: The RAG pipeline

### Steps 1.1 and 1.2: Knowledge base

Every `.md` file in the knowledge base is one document.

* `# ` is the title of the document.
* `## ` starts a new section. Each section is about one topic.
* **A paragraph** is a block of text with a blank line before and after it.

In [ ]:
def load_paragraphs(kb_dir):
    paragraphs = []
    for path in sorted(Path(kb_dir).glob("*.md")):
        title = path.stem
        section = "General"
        for block in re.split(r"\n\s*\n", path.read_text(encoding="utf-8")):
            block = block.strip()
            if not block:
                continue
            if block.startswith("# "):
                title = block[2:].strip()
            elif block.startswith("## "):
                section = block[3:].strip()
            else:
                text = " ".join(block.split())
                paragraphs.append({"doc": path.name, "title": title, "section": section,
                                   "text": text, "words": len(text.split())})
    return paragraphs


paragraphs = load_paragraphs(KB_DIR)
word_counts = [p["words"] for p in paragraphs]
kb_table = pd.DataFrame(paragraphs)

print(f"Documents: {kb_table['doc'].nunique()}")
print(f"Sections: {kb_table[['doc', 'section']].drop_duplicates().shape[0]}")
print(f"Paragraphs: {len(paragraphs)} (minimum 20, recommended 30 to 50)")
print(f"Total words: {sum(word_counts)} (minimum 2,000)")
print(f"Paragraph words: min {min(word_counts)}, max {max(word_counts)} (target 80 to 200)")

outside = kb_table[(kb_table["words"] < 80) | (kb_table["words"] > 200)]
print(f"Paragraphs outside 80 to 200 words: {len(outside)}")
if not USING_SAMPLE and (len(paragraphs) < 20 or sum(word_counts) < 2000):
    print("WARNING: the knowledge base is below the assignment minimum.")

### Step 1.2: Three ways to chunk

All three follow the rules from the assignment. We never cut a sentence in half, and we never put
two sections in the same chunk, because each section is a different topic.

1. **Fixed length:** adds whole sentences to a chunk until it reaches about 120 words, then starts
   the next chunk. It ignores paragraph breaks. A new section also starts a new chunk.
2. **Overlapping paragraphs:** paragraph 1 + 2, then 2 + 3, and so on inside a section. Each chunk
   shares one full paragraph with the next chunk. That shared paragraph is the overlap.
3. **Hybrid:** keeps paragraphs whole. A very short paragraph gets joined to the one next to it,
   and a very long one gets split at the end of a sentence. We also add the document title and
   section name to the text before embedding it, so the chunk still says what it is about. The
   model only sees the normal text.

In [ ]:
FIXED_WORDS = 120
HYBRID_MIN_WORDS = 60
HYBRID_MAX_WORDS = 200
HYBRID_SPLIT_WORDS = 150


def split_sentences(text):
    protected = re.sub(r"\b(e\.g|i\.e|etc|vs)\.", lambda m: m.group(0).replace(".", "<dot>"), text)
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", protected)
    return [p.replace("<dot>", ".").strip() for p in parts if p.strip()]


def group_by_section(paragraphs):
    sections = {}
    for p in paragraphs:
        sections.setdefault((p["doc"], p["title"], p["section"]), []).append(p["text"])
    return sections


def pack_sentences(sentences, limit):
    groups, current, count = [], [], 0
    for sentence in sentences:
        size = len(sentence.split())
        if current and count + size > limit:
            groups.append(" ".join(current))
            current, count = [], 0
        current.append(sentence)
        count += size
    if current:
        groups.append(" ".join(current))
    return groups


def make_chunk(key, text, embed_text=None):
    doc, title, section = key
    return {"doc": doc, "section": section, "text": text, "embed_text": embed_text or text,
            "words": len(text.split())}


def fixed_chunks(paragraphs):
    chunks = []
    for key, texts in group_by_section(paragraphs).items():
        sentences = [s for text in texts for s in split_sentences(text)]
        chunks += [make_chunk(key, text) for text in pack_sentences(sentences, FIXED_WORDS)]
    return chunks


def overlap_chunks(paragraphs):
    chunks = []
    for key, texts in group_by_section(paragraphs).items():
        if len(texts) == 1:
            chunks.append(make_chunk(key, texts[0]))
        for first, second in zip(texts, texts[1:]):
            chunks.append(make_chunk(key, first + " " + second))
    return chunks


def hybrid_chunks(paragraphs):
    chunks = []
    for key, texts in group_by_section(paragraphs).items():
        doc, title, section = key
        merged = []
        for text in texts:
            if merged and len(merged[-1].split()) < HYBRID_MIN_WORDS:
                merged[-1] += " " + text
            else:
                merged.append(text)
        if len(merged) > 1 and len(merged[-1].split()) < HYBRID_MIN_WORDS:
            last = merged.pop()
            merged[-1] += " " + last

        for text in merged:
            pieces = [text]
            if len(text.split()) > HYBRID_MAX_WORDS:
                pieces = pack_sentences(split_sentences(text), HYBRID_SPLIT_WORDS)
            for piece in pieces:
                chunks.append(make_chunk(key, piece, f"{title} - {section}: {piece}"))
    return chunks


CHUNKERS = {"fixed": fixed_chunks, "overlap": overlap_chunks, "hybrid": hybrid_chunks}
chunk_sets = {name: chunker(paragraphs) for name, chunker in CHUNKERS.items()}

pd.DataFrame([
    {"strategy": name, "chunks": len(chunks),
     "avg_words": round(np.mean([c["words"] for c in chunks]), 1),
     "min_words": min(c["words"] for c in chunks),
     "max_words": max(c["words"] for c in chunks)}
    for name, chunks in chunk_sets.items()
])

### Step 1.3: Embeddings

An embedding turns text into a list of numbers. Text with a similar meaning gets similar numbers.
We use three free sentence-transformers models of different sizes:

| Size | Model | Dimensions |
| --- | --- | --- |
| Small | all-MiniLM-L6-v2 | 384 |
| Medium | all-mpnet-base-v2 | 768 |
| Large | BAAI/bge-large-en-v1.5 | 1024 |

The bge model says to add a short line in front of each query, so `query_prefix` does that.

`VectorIndex` keeps all the embeddings in one NumPy matrix. The vectors are normalized, so a dot
product gives us cosine similarity. Our knowledge base is small, so we don't need a vector
database. One matrix multiplication searches everything.

In [ ]:
EMBEDDING_MODELS = {
    "small": {"id": "sentence-transformers/all-MiniLM-L6-v2", "dim": 384, "query_prefix": ""},
    "medium": {"id": "sentence-transformers/all-mpnet-base-v2", "dim": 768, "query_prefix": ""},
    "large": {"id": "BAAI/bge-large-en-v1.5", "dim": 1024,
              "query_prefix": "Represent this sentence for searching relevant passages: "},
}
GENERATION_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
MAX_NEW_TOKENS = 160
TOP_K = 3


class RealEmbedder:
    def __init__(self, config, device):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(config["id"], device=device)
        self.query_prefix = config["query_prefix"]

    def encode_chunks(self, texts):
        return self.model.encode(texts, normalize_embeddings=True, batch_size=32,
                                 show_progress_bar=False)

    def encode_query(self, query):
        return self.model.encode([self.query_prefix + query], normalize_embeddings=True,
                                 show_progress_bar=False)[0]


class FakeEmbedder:
    # Dry run only: hashes words into a vector so the code path runs without a download.
    def __init__(self, config, device=None):
        self.dim = config["dim"]

    def _vector(self, text):
        vector = np.zeros(self.dim)
        for word in re.findall(r"[a-z]+", text.lower()):
            vector[sum(ord(c) * 31 ** i for i, c in enumerate(word)) % self.dim] += 1
        return vector / (np.linalg.norm(vector) or 1)

    def encode_chunks(self, texts):
        return np.array([self._vector(t) for t in texts])

    def encode_query(self, query):
        return self._vector(query)


class VectorIndex:
    def __init__(self, embedder, chunks):
        self.embedder = embedder
        self.chunks = chunks
        started = time.perf_counter()
        self.matrix = np.asarray(embedder.encode_chunks([c["embed_text"] for c in chunks]))
        self.build_seconds = time.perf_counter() - started

    def search(self, query, k):
        started = time.perf_counter()
        query_vector = np.asarray(self.embedder.encode_query(query))
        scores = self.matrix @ query_vector
        top = np.argsort(-scores)[:k]
        search_ms = (time.perf_counter() - started) * 1000
        hits = [{**self.chunks[i], "score": float(scores[i]), "rank": rank}
                for rank, i in enumerate(top, start=1)]
        return hits, search_ms

### Step 1.4: Retrieval

This runs one query so you can see each step: turn the query into an embedding, compare it to every
chunk, and keep the top k. The device setup is the same as A6 and A7. Our 7900 XTX uses AMD's ROCm
version of PyTorch, and it still shows up as `cuda`.

In [ ]:
if RUN_MODE == "dry":
    DEVICE = "cpu"
else:
    import torch

    if torch.cuda.is_available():
        DEVICE = "cuda"
        print("GPU:", torch.cuda.get_device_name(0))
    elif torch.backends.mps.is_available():
        DEVICE = "mps"
    else:
        DEVICE = "cpu"
print("Device:", DEVICE)

embedders = {}
for size, config in EMBEDDING_MODELS.items():
    embedders[size] = (FakeEmbedder if RUN_MODE == "dry" else RealEmbedder)(config, DEVICE)
    dim = len(embedders[size].encode_query("test"))
    if dim != config["dim"]:
        raise ValueError(f"{config['id']} gave {dim} dimensions, expected {config['dim']}")
    print(f"{size}: {config['id']} ({dim} dimensions)")

queries = json.loads(QUERY_FILE.read_text(encoding="utf-8"))
if RUN_MODE == "quick":
    queries = queries[:2]

example = VectorIndex(embedders["small"], chunk_sets["hybrid"])
hits, search_ms = example.search(queries[0]["query"], TOP_K)
print("\nQuery:", queries[0]["query"], f"({search_ms:.1f} ms)")
for hit in hits:
    print(f"  {hit['rank']}. score {hit['score']:.3f} | {hit['doc']} > {hit['section']}")

### Step 1.5: Generation

The prompt has the three parts the assignment asks for: a system instruction, the retrieved chunks
and the question. Loading and generating work the same way as `data_quality/ai/local_model.py`:
16-bit on the GPU, no random sampling, and the model's own chat template.

In [ ]:
SYSTEM_PROMPT = (
    "You answer questions about DataPact, a data quality tool. "
    "Use only the context below. "
    "If the context does not contain the answer, say: I could not find that in the documents."
)


def build_messages(query, hits, system_prompt=SYSTEM_PROMPT):
    context = "\n\n".join(f"[{hit['rank']}] {hit['text']}" for hit in hits)
    return [{"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}]


class RealGenerator:
    def __init__(self, model_id, device):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        kwargs = {"dtype": torch.float16 if device != "cpu" else torch.float32}
        if device != "cpu":
            kwargs["device_map"] = device
        self.model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
        self.model.eval()

    def generate(self, messages):
        inputs = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to(self.model.device)
        with self.torch.no_grad():
            output = self.model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                         pad_token_id=self.tokenizer.pad_token_id)
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


class FakeGenerator:
    # Dry run only: repeats the first sentence of the top chunk.
    def generate(self, messages):
        context = messages[1]["content"].split("\n\nQuestion:")[0]
        return split_sentences(context.split("] ", 1)[1])[0]


generator = FakeGenerator() if RUN_MODE == "dry" else RealGenerator(GENERATION_MODEL, DEVICE)

# The first call is slow while the GPU warms up, so it is not timed in the experiments.
print(generator.generate(build_messages(queries[0]["query"], hits)))

## Part 2: Experiments

### Step 2.1: Test queries

The queries are in `test_queries.json`. Each one says where the answer is and which phrases a good
answer should have, so every setup gets graded the same way. If a query has `"out_of_scope": true`,
the answer is not in our documents, so the right move is for the model to say that.

In [ ]:
pd.DataFrame(queries)

### Step 2.2: Evaluation table

Every setup answers every query. The `_auto` columns are a first guess based on simple rules:

* **context_quality_auto:** Good means we found the right section and at least 75% of the expected
  phrases. Partial means we only got one of those, or at least half the phrases. Anything else is
  Poor.
* **answer_quality_auto:** same idea, but it checks the answer. Good is 75% of the phrases or more,
  and Partial is 40% or more. For an out-of-scope query, Good means the model said it didn't know
  and Poor means it made something up.

Matching phrases is rough. It can mark a right answer as wrong, or a wrong answer as right. So
`context_quality_manual`, `answer_quality_manual` and `notes` are left blank for us to fill in after
reading the answers. The full retrieved chunks are saved too.

In [ ]:
REFUSAL = re.compile(r"could not find|cannot find|can't find|not in the (context|documents)"
                     r"|(does|do) not (contain|say|mention|provide|specify)"
                     r"|not (mentioned|specified|provided|stated|covered|included)|no information", re.I)


def phrase_recall(text, phrases):
    if not phrases:
        return float("nan")
    text = text.lower()
    return sum(p.lower() in text for p in phrases) / len(phrases)


def is_gold(hit, query):
    return hit["doc"] == query.get("expected_doc") and hit["section"] == query.get("expected_section")


def score_retrieval(query, hits):
    if query.get("out_of_scope"):
        return {"context_hit": None, "first_hit_rank": None, "context_phrase_recall": None,
                "context_quality_auto": "n/a"}
    ranks = [hit["rank"] for hit in hits if is_gold(hit, query)]
    recall = phrase_recall(" ".join(hit["text"] for hit in hits), query.get("expected_keywords", []))
    hit = bool(ranks)
    if hit and recall >= 0.75:
        label = "Good"
    elif hit or recall >= 0.5:
        label = "Partial"
    else:
        label = "Poor"
    return {"context_hit": int(hit), "first_hit_rank": min(ranks) if ranks else None,
            "context_phrase_recall": round(recall, 2), "context_quality_auto": label}


def score_answer(query, answer):
    declined = bool(REFUSAL.search(answer))
    if query.get("out_of_scope"):
        return {"answer_phrase_recall": None, "declined": declined,
                "answer_quality_auto": "Good" if declined else "Poor"}
    recall = phrase_recall(answer, query.get("expected_keywords", []))
    label = "Good" if recall >= 0.75 else "Partial" if recall >= 0.4 else "Poor"
    return {"answer_phrase_recall": round(recall, 2), "declined": declined,
            "answer_quality_auto": label}


def run_query(index, query, k=TOP_K, system_prompt=SYSTEM_PROMPT):
    hits, search_ms = index.search(query["query"], k)
    started = time.perf_counter()
    answer = generator.generate(build_messages(query["query"], hits, system_prompt))
    generate_seconds = time.perf_counter() - started
    row = {"query_id": query["id"], "query": query["query"], **score_retrieval(query, hits),
           "answer": answer, **score_answer(query, answer),
           "search_ms": round(search_ms, 1), "generate_s": round(generate_seconds, 2),
           "total_s": round(search_ms / 1000 + generate_seconds, 2)}
    return row, hits


indexes = {}
rows, chunk_rows, index_rows = [], [], []
for size in EMBEDDING_MODELS:
    for strategy, chunks in chunk_sets.items():
        index = VectorIndex(embedders[size], chunks)
        indexes[(size, strategy)] = index
        index_rows.append({"embedding_model": size, "chunking_strategy": strategy, "chunks": len(chunks),
                           "dimensions": index.matrix.shape[1],
                           "vectors_mb": round(index.matrix.nbytes / 1e6, 3),
                           "build_seconds": round(index.build_seconds, 3)})
        for query in queries:
            row, hits = run_query(index, query)
            rows.append({"embedding_model": size, "chunking_strategy": strategy, **row,
                         "retrieved_chunks": "\n---\n".join(
                             f"[{h['rank']}] ({h['score']:.3f}) {h['doc']} > {h['section']}: {h['text']}"
                             for h in hits),
                         "context_quality_manual": "", "answer_quality_manual": "", "notes": ""})
            chunk_rows += [{"embedding_model": size, "chunking_strategy": strategy,
                            "query_id": query["id"], "rank": h["rank"], "score": round(h["score"], 4),
                            "doc": h["doc"], "section": h["section"], "is_expected": is_gold(h, query),
                            "chunk_text": h["text"]} for h in hits]
        print(f"done: {size} + {strategy} ({index.build_seconds:.2f}s to embed {len(chunks)} chunks)")

evaluation = pd.DataFrame(rows)
evaluation.to_csv(OUT_DIR / "evaluation_table.csv", index=False)
pd.DataFrame(chunk_rows).to_csv(OUT_DIR / "retrieved_chunks.csv", index=False)
pd.DataFrame(index_rows).to_csv(OUT_DIR / "index_stats.csv", index=False)
evaluation[["embedding_model", "chunking_strategy", "query_id", "context_quality_auto",
            "answer_quality_auto", "total_s"]].head(15)

### Steps 2.3 and 2.4: Comparing the models and chunking

These tables show averages for each setup, each embedding model and each chunking strategy.
Out-of-scope queries only count toward `declined_correctly`.

In [ ]:
def summarize(table, group):
    in_scope = table[table["context_quality_auto"] != "n/a"]
    out_of_scope = table[table["context_quality_auto"] == "n/a"]
    summary = in_scope.groupby(group).agg(
        context_hit_rate=("context_hit", "mean"),
        context_phrase_recall=("context_phrase_recall", "mean"),
        answer_phrase_recall=("answer_phrase_recall", "mean"),
        avg_search_ms=("search_ms", "mean"),
        avg_generate_s=("generate_s", "mean"),
    ).round(3)
    if len(out_of_scope):
        summary["declined_correctly"] = out_of_scope.groupby(group)["declined"].mean().round(3)
    return summary


by_config = summarize(evaluation, ["embedding_model", "chunking_strategy"])
by_model = summarize(evaluation, "embedding_model")
by_strategy = summarize(evaluation, "chunking_strategy")

by_config.to_csv(OUT_DIR / "summary_by_config.csv")
by_model.to_csv(OUT_DIR / "summary_by_model.csv")
by_strategy.to_csv(OUT_DIR / "summary_by_strategy.csv")

ranked = by_config.sort_values(["context_hit_rate", "answer_phrase_recall", "avg_search_ms"],
                               ascending=[False, False, True])
BEST = ranked.index[0]
print("Best configuration:", BEST)
display(by_model)
display(by_strategy)
ranked

### Step 2.5: Data scaling

This uses the best setup from above. The chunks with the answer stay in every test, and we keep
25%, 50% and then 100% of the other chunks. The answer is always there, so any change comes from
the extra text getting in the way.

* **hit:** how often the right section is in the top k.
* **noise:** how much of what we retrieved came from the wrong section.
* **margin:** the best right chunk's score minus the best wrong chunk's score. A smaller margin
  means the wrong text is getting closer. Below zero means the wrong text won.

In [ ]:
SCALE_LEVELS = [0.25, 0.5, 1.0]
in_scope_queries = [q for q in queries if not q.get("out_of_scope")]
best_size, best_strategy = BEST
all_chunks = chunk_sets[best_strategy]
rng = np.random.default_rng(42)
shuffled = [all_chunks[i] for i in rng.permutation(len(all_chunks))]

scale_rows = []
for level in SCALE_LEVELS:
    keep = shuffled[:max(1, round(len(shuffled) * level))]
    for query in in_scope_queries:
        gold = [c for c in all_chunks if is_gold(c, query)]
        chunks = gold + [c for c in keep if not is_gold(c, query)]
        index = VectorIndex(embedders[best_size], chunks)
        hits, _ = index.search(query["query"], TOP_K)
        all_hits, _ = index.search(query["query"], len(chunks))
        gold_scores = [h["score"] for h in all_hits if is_gold(h, query)]
        other_scores = [h["score"] for h in all_hits if not is_gold(h, query)]
        margin = max(gold_scores) - max(other_scores) if gold_scores and other_scores else None
        scale_rows.append({"share_of_data": level, "chunks": len(chunks), "query_id": query["id"],
                           "hit": int(any(is_gold(h, query) for h in hits)),
                           "noise": 1 - sum(is_gold(h, query) for h in hits) / len(hits),
                           "margin": margin})

scaling = pd.DataFrame(scale_rows)
scaling.to_csv(OUT_DIR / "scaling_results.csv", index=False)
scaling.groupby("share_of_data")[["chunks", "hit", "noise", "margin"]].mean().round(3)

## Part 3: Failure analysis and improvement

### Step 3.1: Failure cases

These are the worst rows from the evaluation table, with their chunks and answers. Read them and
pick at least 3 real failures for `rag_analysis.md`. Some may turn out fine, since the auto score
can be wrong.

In [ ]:
RANK = {"Poor": 0, "Partial": 1, "Good": 2, "n/a": 3}
failures = evaluation.assign(
    context_rank=evaluation["context_quality_auto"].map(RANK),
    answer_rank=evaluation["answer_quality_auto"].map(RANK),
).sort_values(["answer_rank", "context_rank"])
failures = failures[(failures["answer_rank"] < 2) | (failures["context_rank"] < 2)]
failures.drop(columns=["context_rank", "answer_rank"]).to_csv(OUT_DIR / "failure_candidates.csv", index=False)
print("Failure candidates:", len(failures))

for _, row in failures.head(5).iterrows():
    print("=" * 80)
    print(f"{row['embedding_model']} + {row['chunking_strategy']} | {row['query']}")
    print(f"context: {row['context_quality_auto']} | answer: {row['answer_quality_auto']}")
    print("ANSWER:", row["answer"])
    print("CHUNKS:\n" + row["retrieved_chunks"])

### Step 3.3: Improvement attempt

Each test uses the best setup and every query. `baseline` is the "before" result. The others change
one thing: how many chunks we retrieve, or how strict the prompt is.

In [ ]:
STRICT_PROMPT = (
    "You answer questions about DataPact, a data quality tool. "
    "Answer in one or two sentences, using only the numbered context. "
    "Do not add facts that are not in the context. "
    "If the context does not contain the answer, reply only with: "
    "I could not find that in the documents."
)

VARIANTS = {
    "baseline": {"k": TOP_K, "system_prompt": SYSTEM_PROMPT},
    "top_k_1": {"k": 1, "system_prompt": SYSTEM_PROMPT},
    "top_k_5": {"k": 5, "system_prompt": SYSTEM_PROMPT},
    "strict_prompt": {"k": TOP_K, "system_prompt": STRICT_PROMPT},
}

variant_rows = []
for name, settings in VARIANTS.items():
    for query in queries:
        row, _ = run_query(indexes[BEST], query, **settings)
        variant_rows.append({"variant": name, **row})

variants = pd.DataFrame(variant_rows)
variants.to_csv(OUT_DIR / "improvement_results.csv", index=False)
summarize(variants, "variant")

## Results files

| File | What is in it |
| --- | --- |
| `evaluation_table.csv` | Step 2.2: one row per setup and query, with full chunks and answers |
| `retrieved_chunks.csv` | The same chunks with one chunk per row |
| `index_stats.csv` | Number of chunks, storage size and build time per setup, for the cost question in Step 4.1 |
| `summary_by_config.csv`, `summary_by_model.csv`, `summary_by_strategy.csv` | Steps 2.3 and 2.4 |
| `scaling_results.csv` | Step 2.5 |
| `failure_candidates.csv` | Step 3.1 |
| `improvement_results.csv` | Step 3.3 |